# Recurrent Neural Network: Từ RNN đến LSTM

## 1. Introdution

Đối với các bạn học deep learning thì không thể không biết tới RNN, một thuật toán cực kì quan trọng chuyên xử lý thông tin dạng chuỗi. Đầu tiên, hãy nhìn xem RNN có thể làm gì. Dưới đây là một vài ví dụ:
- Machine Translation (Dịch máy)
- Mô hình hóa ngôn ngữ và sinh văn bản: đây có lẽ là khả năng ấn tượng nhất đối với mình.
- Nhận dạng giọng nói
- Mô tả hình ảnh: RNN kết hợp cùng CNN để sinh ra mô tả cho hình ảnh chưa được gán nhãn. Đây cũng là một bài tập khá hay mà mình sẽ giới thiệu trong bài viết tiếp theo.

## 2. Recurrent Neural Network

### Ý tưởng căn bản

Một cách nôm na, đối với mạng neural thông thường, chúng ta cho tất cả dữ liệu vào cùng một lúc. Nhưng đôi khi, dữ liệu của chúng ta mang ý nghĩa trình tự, tức nếu thay đổi trình tự dữ liệu, kết quả sẽ khác.

Dễ thấy rõ nhất ở dữ liệu văn bản. Ví dụ, “Con ăn cơm chưa” và “Con chưa ăn cơm”, nếu tách mỗi câu theo từ, ta được bộ vocab [ ‘con’, ‘ăn’, ‘cơm’, ‘chưa’], one hot encoding và cho tất cả vào mạng neural , có thể thấy ngay, không có sự phân biệt nào giữa 2 câu trên. Việc đảo thứ tự duyệt các từ làm sai lệch ý nghĩ của câu.

Nói cách khác, chúng ta cần một mạng neural có thể xử lí tuần tự.

Vậy làm sao để xử lí tuần tự, đầu tiên cần đưa đầu vào vào một cách tuần tự

Mình là kiểu người hiểu nhanh hơn thông qua hình ảnh 😊, và mình nghĩ đây là hình ảnh thể hiện rõ ràng nhất rốt cuộc RNN làm gì. Mỗi block RNN sẽ lấy thông tin từ các block trước và input hiện tại.

![](image1.png)

Các x ở đây đại diện cho dữ liệu đầu vào lần lượt (được chia theo time step).

$x_{t}$ đại diện cho time step thứ $t$, và $y_{t}$ là output của một step. Ví dụ $x_{2}$ sẽ là vector đại diện từ thứ 2 trong câu văn bản.

Hình ảnh dưới đây cho thấy rõ hơn điều gì thực sự xảy ra trong một step.

![](image2.png)

- Hidden State: $h_{t}$. Đây chính là bộ nhớ của mạng. $h_{t}$ là tổng hợp thông tin của hidden state trước ($h_{t-1}$) + input tại time step $t$ ($x_{t}$). Activation function ở đây là $g_{1}$ (chủ yếu là $\text{tanh}$ hoặc $\text{ReLU}$).

$$
h_{t} = g_{1}(W_{hh}.h_{t-1} + W_{hx}.x_{t} + b_{h})
$$

$\rightarrow$ Hoặc có thể viết gọn hơn:
$$
h_{t} = g_{1}((W_{hh}W_{hx})(h_{t-1} x_{t}))
$$
$$
h_{t} = g_{1}((W)(h_{t-1}x_{t}))
$$

- Output của từng time step $y_{t}$: Tại 1 block của mạng RNN có 2 đầu ra. Trong đó, $h_{t}$ là tổng hợp thông tin của các state trước để tiếp tục truyền đi trong chuỗi mạng, và ta có thêm $y_{t}$ là output của từng time step một. Ở đây $g_{2}$ thường là hàm $\text{softmax}$.
$$
y_{t} = g_{2}(W_{yh}.h_{t} + b_{y})
$$

### Tính toán lan truyền ngược (BPTT - Backpropagation Through Time)

Như vậy, trong quá trình training, có 3 tham số chúng ta cần tìm $W_{hh}, W_{hy}, W{hx}$.

Chúng ta cần tính $\frac{\partial L}{\partial W_{hh}}$, $\frac{\partial L}{\partial W_{hx}}$, $\frac{\partial L}{\partial W_{hy}}$. (Với $L$ là loss function).

![](image3.png)

Nhìn chung, ta có thể thấy vấn đề cơ bản ở đây là:

Trong NN truyền thống, ta không chia sẻ tham số giữa các tầng mạng, tuy vậy, với RNN, ta có thể thayas, để tính đạo hàm của loss the $W_{hh}$, ta phụ thuộc vào $h_{t-1}$, mà $h_{t-1}$ lại phụ thuộc vào $h_{t-2}$ và $x_{t-1}$. Nói nôm na, ta phỉa cộng tất cả đầu ra ở các bước trước để tính đạo hàm. Điều này gây ra một hạn chế lớn cho RNN:
- Hiện tượng $\text{vanishing gradient}$(không học được long-term dependency) / $\text{exploding gradient}$(training không ổn định).
- Kéo theo chi phí tính toán cực lớn khi $\text{BPTT}$.

Như vậy, so với mạng NN bình thường, RNN có thể nắm bắt thông tin dạng chuỗi. Điều này cũng góp phần giúp RNN có thể đáp ứng chuỗi đầu vào có độ dài tùy ý, kích thước model không bị phụ thuộc vào size đầu vào.

### Hạn chế của RNN là gì?

- Phải thực hiện tuần tự: Không tận dụng được khả năng tính toán song song của máy tính (GPU/TPU).
- Vanishing Gradient (Đạo hàm bị triệt tiêu).
    - Vì hàm kích hoạt ($\text{tanh}$ hay $\text{sigmoid}$) của ta sẽ cho kết quả đầu ra nằm trong đoạn [-1,1] (với sigmoid là [0,1]) nên đạo hàm của no sẽ đóng trong khoảng [0,1] (với sigmoid là [0,0.25]).
    - Ở trên, chúng ta đã dùng chain rule để tính đạo hàm. Có một vấn đề ở đây là, hàm tanh lẫn sigmoid đều có đạo hàm bằng 0 tại 2 đầu. Mà khi đạo hàm bằng 0 thì nút mạng tương ứng tại đó sẽ bị bão hòa. Lúc đó các nút phía trước cũng sẽ bị bão hoà theo. Nên với các giá trị nhỏ trong ma trận, khi ta thực hiện phép nhân ma trận sẽ đạo hàm tương ứng sẽ xảy ra Vanishing gradient, tức đạo hàm bị triệt tiêu chỉ sau vài bước nhân. Như vậy, các bước ở xa sẽ không còn tác dụng với nút hiện tại nữa, làm cho RNN không thể học được các phụ thuộc xa. Vấn đề này không chỉ xảy ra với mạng RNN mà ngay cả mạng neural truyền thống với nhiều lớp cũng có hiện tượng này.
    - Với cách nhìn như trên, ngoài Vanishing gradient, ta còn gặp phải Exploding Gradient (bùng nổ đạo hàm). Tùy thuộc vào hàm kích hoạt và tham số của mạng, vấn đề này xảy ra khi các giá trị của ma trận là lớn (lớn hơn 1). Tuy nhiên, người ta thường nói về vấn đề Vanishing nhiều hơn là Exploding, vì 2 lý do sau:
        - Thứ nhất, bùng nổ đạo hàm có thể theo dõi được vì khi đạo hàm bị bùng nổ thì ta sẽ thu được kết quả là một giá trị phi số NaN làm cho chương trình của ta bị dừng hoạt động.
        - Thứ hai, bùng nổ đạo hàm có thể ngăn chặn được khi ta đặt một ngưỡng giá trị trên (tham khảo kỹ thuật Gradient Clipping). Còn rất khó để theo dõi sự mất mát đạo hàm cũng như tìm cách xử lí nó.

Để xử lý Vanishing Gradient, có 2 cách phổ biến:
- Cách thứ nhất, thay vì sử dụng activation function là tanh và sigmoid, ta thay bằng ReLu (hoặc các biến thể như Leaky ReLu). Đạo hàm của ReLu hoặc là 0 hoặc là 1, nên ta có thể kiểm soát phần nào vấn đề mất mát đạo hàm.
- Cách thứ hai, ta thấy RNN thuần không hề có thiết kế nào để lọc đi những thông tin không cần thiết. Ta cần thiết kế một kiến trúc có thể nhớ dài hạn hơn, đó là LSTM và GRU.

## 3. LSTM (Long Short-Term Memory)

Mình nghĩ hình ảnh sau đây là rõ nét nhất để so sánh giữa RNN và LSTM.

### RNN

![](image4.png)

### LSTM

![](image5.png)

Về cơ bản, ý tưởng không khác nhau là mấy. Chúng ta chỉ thêm một số tính toán ở đây. Tất cả được tóm tắt trong hình sau

![](image6.png)

Đâuf tiên, chúng ta có $i,f,g$ có công thức gần giống hệt nhau và chỉ khác mỗi ma trận tham số. Chính ma trậnnayf sẽ quyết định chức năng khác nhau của từng cổng. $\sigma$ là ký hiệu của hàm sigmoid. 

Giải thích kiến trúc:
- Input Gate $i$: Cổng vào
    - Cổng vào giúp quyết định bao nhiêu thông tin đầu vào sẽ ảnh hưởng đến trạng thái mới. Quyết định bằng cách nào, thông qua đặc điểm của hàm sigmoid (đầu ra nằm trong khoảng [0,1]), như vậy khi một vector thông tin đi qua đây, nếu nâhn với 0, vector sẽ bị triệt tiêu hoàn toàn. Nếu nhân với 1, hầu hết thông tínex được giữ lại.

- Tương tự như vậy, $f$ là $\text{forget gate}$ - cổng quên.
    - Cổng quyết định sẽ bỏ đi bao nhiêu lượng thông tin đến từ trạng thái trước đó.

- Cuối cùng, cổng $o$ là $\text{output gate}$ - cổng ra.
    - Cổng điều chỉnh lượng thông ttin có thể ra ngoài $y_{t}$ và lượng thông tin truyền tới trạng thái tiếp theo.

- Tiếp theo, $g$ thực chất cũng chỉ là một trạng thái ẩn được tính dựa trên đầu vào hiện tại $x_{t}$ và trạng thái trước $h_{t-1}$. Tính hàn toàn tương tự như input gate, chỉ thay vì dùng $\text{sigmoid}$, ta dùng $\text{tanh}$. Kết hợp hai điều này lại đeer cập nhật trạng thái mới.

- Cuối cùng, ta có $c_{t}$ là bộ nhớ trong của LSTM. Nhìn vào công thứcc, có thể thấy nó là tổng hợp của bộ nhớ trước $c_{t-1}$ đã được lọc qua cổng quên $f$, cộng với trạng thái ẩn $g$ đã được lọc bởi cổng vào $i$. Cell state sẽ mang thông tin nào quan trọng truyền đi xa hơn và sẽ được dùng khi cần. Đây chính là $\text{long term memory}$.

- Sau khi có $c_{t}$, ta sẽ đưa nó qua cổng ra để lọc thông tin một lần nữa, thu được trạng thái mới $h_{t}$.

Nếu nhìn kỹ một chút, ta có thể thấy RNN truyền thống là dạng đặc biệt của LSTM. Nếu thay giá trị đầu ra của input gate là 1 và đầu ra forget gate là 0 (không nhớ trạng thái trước), ta được RNN thuần.

## Tổng kết

Nhìn một lượt qua kiến trúc LSTM, ta có thể tóm tắt:
- Thứ nhất, LSTM có long-term memmory. Tuy nhiên, $h_{t}$, $g_{t}$ khá giống với RNN truyền thống, tức có short-term memory. Nhìn chung, LSTM giải quyết phần nào vanishing gradient so với RNN, nhưng chỉ một phần.
- Với lượng tính toán như trên, RNN đã chậm, LSTM nay còn chậm hơn.

Tuy vậy, với những cải tiến so với RNN thuần, LSTM đã và đang được sử dụng phổ biến. Trên thực tế, cách cài đặt LSTM cũng rất đa dạng và linh hoạt theo bài toán, tuy nhiên vẫn dựa trên LSTM chuẩn như trên.